# Regression with an Abalone Dataset 참가 보고서

본 보고서는 Kaggle Playground Series Season 4 Episode 4 대회인 **Regression with an Abalone Dataset**에 참가하여, 데이터 분석부터 모델 학습, 예측 파일 생성, 제출 결과 분석까지의 과정을 정리한 것이다.

이 대회는 전복의 성별과 물리적 측정값을 이용하여 `Rings` 값을 예측하는 정형 데이터 회귀 문제이다. `Rings`는 전복 껍질의 고리 수를 의미하며, 전복의 나이와 직접적으로 관련된 값이다.

## 1. 대회 개요

Regression with an Abalone Dataset은 전복의 물리적 측정값을 바탕으로 `Rings` 값을 예측하는 정형 데이터 회귀 대회이다. 입력 데이터는 전복의 성별, 길이, 지름, 높이, 무게 관련 변수들로 구성되어 있으며, 목표 변수는 `Rings`이다.

전복의 나이를 실제로 측정하려면 껍질을 자르고 염색한 뒤 현미경으로 고리 수를 세는 과정이 필요하다. 이 과정은 시간이 오래 걸리고 번거롭기 때문에, 전복의 외형적·물리적 특성만으로 `Rings`를 예측하는 것이 이 대회의 핵심 목적이다.

문제 유형은 수치형 값을 예측하는 회귀 문제이다. 최종적으로 모델은 `test.csv`에 포함된 각 전복 데이터에 대해 `Rings` 값을 예측하고, Kaggle 제출 형식에 맞는 `submission.csv` 파일을 생성해야 한다.

## 2. 평가 방법

이 대회는 RMSLE(Root Mean Squared Logarithmic Error)를 평가 지표로 사용한다. RMSLE는 실제값과 예측값의 차이를 로그 스케일에서 계산하는 지표이다. 일반적인 RMSE가 실제값과 예측값의 절대적인 차이에 민감하다면, RMSLE는 두 값 사이의 비율 차이를 더 중요하게 반영한다.

$$
\mathrm{RMSLE}
=
\sqrt{
\frac{1}{n}
\sum_{i=1}^{n}
\left(
\log(1+\hat{y}_i) - \log(1+y_i)
\right)^2
}
$$

여기서 \(y_i\)는 실제 `Rings` 값이고, \(\hat{y}_i\)는 모델이 예측한 `Rings` 값이다.

RMSLE는 로그를 사용하기 때문에 예측값이 음수가 되면 계산에 문제가 생긴다. 따라서 최종 예측값은 0보다 작지 않도록 보정해야 한다. 또한 큰 값에서의 절대 오차뿐만 아니라 작은 값에서 비율적으로 크게 벗어나는 예측도 점수에 영향을 준다.

## 3. 대회 규칙 및 제출 방식

제공된 `train.csv`를 이용하여 모델을 학습하고, `test.csv`의 각 `id`에 대해 `Rings` 값을 예측한다. 최종 제출 파일은 `sample_submission.csv`와 동일한 구조를 가져야 하며, `id`와 `Rings` 두 컬럼을 포함해야 한다.

대회 평가는 제출된 `Rings` 예측값과 실제 정답 사이의 RMSLE로 이루어진다. RMSLE는 낮을수록 좋은 점수이므로, 모델 학습 과정에서는 검증 데이터의 RMSLE를 기준으로 성능을 비교한다.

공식 경쟁 기간은 종료되었지만 Late Submission을 통해 동일한 형식으로 제출 파일을 업로드하고 점수를 확인할 수 있다.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error, mean_squared_error, mean_absolute_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 1000)


## 4. 데이터 불러오기

데이터는 `train.csv`, `test.csv`, `sample_submission.csv`로 구성된다.

`train.csv`에는 모델 학습에 사용할 입력 변수와 정답값인 `Rings`가 포함되어 있다. `test.csv`에는 `Rings`가 포함되어 있지 않으며, 이 데이터에 대한 예측값을 생성해야 한다. `sample_submission.csv`는 Kaggle 제출 파일의 기본 형식을 제공한다.

Kaggle Notebook 환경에서는 데이터가 `/kaggle/input/playground-series-s4e4/` 경로에 위치한다. 로컬 Jupyter Notebook에서 실행할 경우에는 `data` 폴더 안에 csv 파일을 넣고 실행한다.

In [ ]:
kaggle_path = "/kaggle/input/playground-series-s4e4"
local_path = "./data"

if os.path.exists(kaggle_path):
    DATA_DIR = kaggle_path
elif os.path.exists(local_path):
    DATA_DIR = local_path
else:
    DATA_DIR = "."

train_path = os.path.join(DATA_DIR, "train.csv")
test_path = os.path.join(DATA_DIR, "test.csv")
sample_path = os.path.join(DATA_DIR, "sample_submission.csv")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_path)

print("train shape:", train.shape)
print("test shape:", test.shape)
print("sample_submission shape:", sample_submission.shape)

## 5. 데이터 기본 구조 확인

학습 데이터와 테스트 데이터의 구조를 확인한다. 학습 데이터에는 `Rings`가 포함되어 있고, 테스트 데이터에는 `Rings`가 제외되어 있다. 따라서 학습 데이터의 입력 변수와 정답값을 이용해 모델을 학습한 뒤, 테스트 데이터의 `Rings` 값을 예측하는 방식으로 진행한다.

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
sample_submission.head()

In [ ]:
train.info()

In [ ]:
test.info()

In [ ]:
train.describe()

`id`는 각 행을 구분하기 위한 식별자이므로 모델 학습에는 직접 사용하지 않는다. `Sex`는 문자열 형태의 범주형 변수이고, 나머지 입력 변수들은 전복의 크기와 무게를 나타내는 수치형 변수이다.

`Rings`는 예측해야 하는 목표 변수이며, 학습 데이터에만 포함되어 있다.

## 6. 결측치 확인

모델 학습 전에 결측치 여부를 확인한다. 결측치가 존재하면 수치형 변수는 평균 또는 중앙값으로 대체하고, 범주형 변수는 최빈값으로 대체하는 방식이 필요하다. 결측치가 없다면 별도의 결측치 처리 없이 다음 단계로 진행한다.

In [ ]:
print("Train missing values")
display(train.isnull().sum())

print("Test missing values")
display(test.isnull().sum())

결측치 확인 결과를 바탕으로 전처리 방향을 결정한다. 결측치가 없는 경우에는 데이터 손실이나 대체값에 따른 왜곡 없이 그대로 모델 학습에 사용할 수 있다.

## 7. 목표 변수 `Rings` 분석

`Rings`는 이 대회에서 예측해야 하는 목표 변수이다. 회귀 문제에서는 목표 변수의 분포를 먼저 확인하는 것이 중요하다. 목표값이 특정 구간에 몰려 있는지, 극단적인 값이 존재하는지에 따라 모델 성능과 예측 안정성이 달라질 수 있다.

또한 이 대회는 RMSLE를 평가 지표로 사용하므로, `Rings` 값의 분포와 로그 변환 가능성을 함께 고려해야 한다.

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(train["Rings"], bins=30)
plt.title("Distribution of Rings")
plt.xlabel("Rings")
plt.ylabel("Count")
plt.show()

train["Rings"].describe()

In [ ]:
train["Rings"].value_counts().sort_index()

`Rings`는 정수 형태의 값을 가지며, 대부분의 데이터가 특정 구간에 집중되어 있다. 일부 큰 값을 가지는 데이터도 존재하지만 전체적으로는 중심 구간에 데이터가 많이 분포한다.

RMSLE는 로그 기반 지표이므로 큰 값의 절대 오차뿐만 아니라 작은 값에서 비율적으로 크게 벗어나는 예측도 중요하다. 따라서 모델 학습 과정에서 예측값이 음수가 되지 않도록 처리하고, 목표 변수에 로그 변환을 적용한 모델도 함께 비교한다.

## 8. 범주형 변수 `Sex` 분석

`Sex`는 전복의 성별을 나타내는 범주형 변수이다. 값은 일반적으로 `M`, `F`, `I`로 구성된다.

- `M`: Male
- `F`: Female
- `I`: Infant

범주형 변수는 모델이 직접 처리하기 어렵기 때문에 학습 단계에서 One-Hot Encoding을 적용한다. 먼저 각 범주의 개수와 범주별 `Rings` 분포를 확인한다.

In [ ]:
train["Sex"].value_counts()

In [ ]:
plt.figure(figsize=(6, 4))
train["Sex"].value_counts().plot(kind="bar")
plt.title("Count of Sex")
plt.xlabel("Sex")
plt.ylabel("Count")
plt.show()

In [ ]:
train.groupby("Sex")["Rings"].agg(["count", "mean", "median", "std", "min", "max"])

성별 그룹에 따라 `Rings`의 평균과 중앙값이 다르게 나타날 수 있다. 특히 Infant에 해당하는 전복은 성장 정도가 낮을 가능성이 있으므로 `Rings` 값이 상대적으로 작게 나타날 수 있다.

따라서 `Sex`는 단순한 구분값이 아니라 `Rings` 예측에 의미 있는 범주형 변수로 사용할 수 있다.

## 9. 수치형 변수 분포 분석

수치형 변수들은 전복의 길이, 지름, 높이, 무게를 나타낸다. 이 변수들은 전복의 성장 정도와 직접적으로 관련될 가능성이 크기 때문에 `Rings` 예측에서 중요한 역할을 한다.

각 변수의 분포를 확인하여 값의 범위, 치우침, 이상치 가능성을 살펴본다.

In [ ]:
numeric_cols = [col for col in train.columns if col not in ["id", "Sex", "Rings"]]

numeric_cols

In [ ]:
for col in numeric_cols:
    plt.figure(figsize=(7, 4))
    plt.hist(train[col], bins=40)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.show()

전복의 크기와 무게 관련 변수들은 대부분 연속형 분포를 보인다. 일부 변수에서는 특정 구간에 데이터가 몰려 있거나, 상대적으로 큰 값을 가지는 데이터가 존재할 수 있다.

이러한 변수들은 모델이 `Rings`를 예측할 때 중요한 입력 정보가 된다. 특히 전체 무게, 껍질 무게, 길이, 지름은 전복의 성장 정도를 반영할 수 있다.

## 10. 입력 변수와 `Rings`의 관계 분석

각 수치형 변수가 `Rings`와 어떤 관계를 가지는지 산점도를 통해 확인한다. 변수 값이 증가할수록 `Rings`가 함께 증가하는 경향이 있다면 해당 변수는 예측에 유용한 변수로 볼 수 있다.

다만 실제 데이터에서는 완전한 직선 관계보다는 비선형적인 관계가 나타날 수 있다. 따라서 선형 모델뿐만 아니라 비선형 관계를 학습할 수 있는 트리 기반 모델도 함께 사용한다.

In [ ]:
for col in numeric_cols:
    plt.figure(figsize=(7, 4))
    plt.scatter(train[col], train["Rings"], alpha=0.2)
    plt.title(f"{col} vs Rings")
    plt.xlabel(col)
    plt.ylabel("Rings")
    plt.show()

산점도를 통해 전복의 크기와 무게가 증가할수록 `Rings`도 증가하는 경향을 볼 수 있다. 하지만 모든 변수가 일정한 직선 형태로 증가하지는 않는다.

따라서 이 문제에서는 단순 선형 회귀만으로는 변수 간 관계를 충분히 반영하기 어렵고, 비선형 관계와 변수 간 상호작용을 학습할 수 있는 모델이 필요하다.

## 11. 상관관계 분석

수치형 변수들 사이의 상관관계와 `Rings`와의 상관관계를 확인한다. 상관계수는 -1에서 1 사이의 값을 가지며, 1에 가까울수록 양의 상관관계가 강하고 -1에 가까울수록 음의 상관관계가 강하다.

상관관계 분석은 어떤 변수가 목표 변수와 관련이 큰지 파악하는 데 도움이 된다.

In [ ]:
corr = train.drop(columns=["id", "Sex"]).corr()

plt.figure(figsize=(10, 8))
plt.imshow(corr, aspect="auto")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.colorbar()
plt.title("Correlation Heatmap")
plt.show()

corr["Rings"].sort_values(ascending=False)

상관관계 분석 결과, 전복의 크기와 무게 관련 변수들이 `Rings`와 관련성을 가진다. 다만 입력 변수들끼리도 서로 높은 상관관계를 보일 수 있다. 예를 들어 길이와 지름, 전체 무게와 각 무게 변수들은 서로 연관되어 있을 가능성이 크다.

상관관계가 높다는 것은 예측에 유용할 수 있다는 의미이지만, 반드시 해당 변수 하나만으로 좋은 예측이 가능하다는 뜻은 아니다. 따라서 여러 변수를 함께 사용하여 모델이 종합적으로 패턴을 학습하도록 한다.

## 12. Feature Engineering

기본 변수만으로도 모델 학습은 가능하지만, 전복의 형태와 구성 비율을 더 잘 표현하기 위해 파생 변수를 추가한다.

전복의 길이, 지름, 높이를 곱하면 부피와 유사한 값을 만들 수 있다. 또한 전체 무게 대비 껍질 무게, 살 무게, 내장 무게의 비율은 전복의 성장 상태를 나타내는 보조 정보로 사용할 수 있다.

추가하는 파생 변수는 다음과 같다.

- `Volume`: Length × Diameter × Height
- `Shell_ratio`: Shell weight / Whole weight
- `Shucked_ratio`: Whole weight.1 / Whole weight
- `Viscera_ratio`: Whole weight.2 / Whole weight
- `Diameter_Length_ratio`: Diameter / Length
- `Height_Length_ratio`: Height / Length

이러한 파생 변수는 단순한 크기 정보뿐만 아니라 전복의 형태와 무게 구성 비율을 반영한다.

In [ ]:
def add_features(df):
    df = df.copy()
    
    eps = 1e-9
    
    df["Volume"] = df["Length"] * df["Diameter"] * df["Height"]
    df["Shell_ratio"] = df["Shell weight"] / (df["Whole weight"] + eps)
    df["Shucked_ratio"] = df["Whole weight.1"] / (df["Whole weight"] + eps)
    df["Viscera_ratio"] = df["Whole weight.2"] / (df["Whole weight"] + eps)
    df["Diameter_Length_ratio"] = df["Diameter"] / (df["Length"] + eps)
    df["Height_Length_ratio"] = df["Height"] / (df["Length"] + eps)
    
    return df

train_fe = add_features(train)
test_fe = add_features(test)

train_fe.head()

## 13. 학습 데이터 준비

모델 학습을 위해 입력 변수와 목표 변수를 분리한다. `id`는 단순 식별자이므로 제거하고, `Rings`는 목표 변수로 따로 분리한다.

`Sex`는 범주형 변수이므로 One-Hot Encoding을 적용한다. 나머지 수치형 변수들은 StandardScaler를 적용하여 변수의 스케일 차이를 줄인다.

In [ ]:
target = "Rings"

X = train_fe.drop(columns=["id", target])
y = train_fe[target]

X_test = test_fe.drop(columns=["id"])

categorical_features = ["Sex"]
numeric_features = [col for col in X.columns if col not in categorical_features]

print("categorical_features:", categorical_features)
print("numeric_features:", numeric_features)

In [ ]:
try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", onehot, categorical_features),
        ("num", StandardScaler(), numeric_features)
    ],
    remainder="drop"
)

## 14. 평가 함수 정의

Kaggle 평가 지표와 동일하게 RMSLE를 계산하는 함수를 정의한다. RMSLE는 로그를 사용하는 지표이므로 예측값이 음수가 되면 안 된다. 따라서 예측값이 0보다 작을 경우 0으로 보정한 뒤 점수를 계산한다.

In [ ]:
def rmsle(y_true, y_pred):
    y_pred = np.maximum(y_pred, 0)
    return np.sqrt(mean_squared_log_error(y_true, y_pred))


## 15. Train / Validation 데이터 분리

Kaggle의 test 데이터는 정답이 공개되어 있지 않기 때문에, train 데이터의 일부를 검증용 데이터로 분리하여 모델 성능을 평가한다.

학습 데이터의 80%는 모델 학습에 사용하고, 나머지 20%는 검증용 데이터로 사용한다. 검증용 데이터에서 계산한 RMSLE를 기준으로 모델 성능을 비교한다.

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)

## 16. 모델 선정

이 문제는 정형 데이터 기반 회귀 문제이므로 선형 모델과 트리 기반 앙상블 모델을 함께 비교한다.

Linear Regression은 가장 기본적인 회귀 모델이므로 baseline으로 사용한다. Ridge Regression은 선형 회귀에 L2 정규화를 적용한 모델로, 과적합을 줄이는 효과가 있다.

Random Forest Regressor는 여러 개의 결정 트리를 학습하고 예측을 평균내는 앙상블 모델이다. 변수 간 비선형 관계를 학습할 수 있고, 단일 결정 트리보다 안정적인 예측이 가능하다.

HistGradientBoostingRegressor는 Gradient Boosting 계열 모델로, 이전 모델의 오차를 보완하는 방식으로 학습한다. 정형 데이터 회귀 문제에서 성능이 좋은 편이며, 학습 속도도 비교적 빠르다.

전복의 크기와 무게가 증가한다고 해서 `Rings`가 항상 일정한 비율로 증가하는 것은 아니므로, 비선형 관계를 학습할 수 있는 트리 기반 모델이 적합할 가능성이 높다. 최종 모델은 검증 데이터에서 RMSLE가 가장 낮은 모델로 선정한다.

In [ ]:
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        max_depth=None
    ),
    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=31,
        random_state=42
    )
}

results = []

for name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_valid)
    pred = np.maximum(pred, 0)
    
    score_rmsle = rmsle(y_valid, pred)
    score_rmse = np.sqrt(mean_squared_error(y_valid, pred))
    score_mae = mean_absolute_error(y_valid, pred)
    score_r2 = r2_score(y_valid, pred)
    
    results.append({
        "model": name,
        "RMSLE": score_rmsle,
        "RMSE": score_rmse,
        "MAE": score_mae,
        "R2": score_r2
    })
    
    print(f"{name} 완료 - RMSLE: {score_rmsle:.5f}")

results_df = pd.DataFrame(results).sort_values("RMSLE")
results_df

## 17. 모델 비교 결과 분석

모델별 검증 성능을 RMSLE 기준으로 비교하였다. RMSLE는 낮을수록 좋은 점수이므로, 결과표에서 RMSLE가 가장 낮은 모델을 우선적으로 선택한다.

선형 모델은 기준 성능을 확인하는 데 의미가 있지만, 변수와 목표값 사이의 관계를 직선 형태로 가정한다는 한계가 있다. 반면 Random Forest와 HistGradientBoostingRegressor는 비선형 관계와 변수 간 상호작용을 학습할 수 있으므로 이 데이터에 더 적합할 가능성이 크다.

검증 데이터에서 가장 낮은 RMSLE를 보인 모델을 최종 후보 모델로 사용한다.

## 18. 로그 변환 모델 실험

이 대회의 평가 지표는 RMSLE이다. RMSLE는 로그 스케일에서 실제값과 예측값의 차이를 계산하므로, 목표 변수인 `Rings`에 로그 변환을 적용해 학습하는 방식도 실험한다.

`log1p`는 \(\log(1+x)\)를 계산하는 함수이고, `expm1`은 변환된 값을 다시 원래 스케일로 복원하는 함수이다. 이를 통해 모델은 로그 변환된 목표값을 학습하고, 예측 결과는 다시 원래 `Rings` 값으로 변환된다.

In [ ]:
log_results = []

for name, model in models.items():
    log_model = TransformedTargetRegressor(
        regressor=model,
        func=np.log1p,
        inverse_func=np.expm1
    )
    
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", log_model)
    ])
    
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_valid)
    pred = np.maximum(pred, 0)
    
    score_rmsle = rmsle(y_valid, pred)
    score_rmse = np.sqrt(mean_squared_error(y_valid, pred))
    score_mae = mean_absolute_error(y_valid, pred)
    score_r2 = r2_score(y_valid, pred)
    
    log_results.append({
        "model": name + "_log_target",
        "RMSLE": score_rmsle,
        "RMSE": score_rmse,
        "MAE": score_mae,
        "R2": score_r2
    })
    
    print(f"{name}_log_target 완료 - RMSLE: {score_rmsle:.5f}")

log_results_df = pd.DataFrame(log_results).sort_values("RMSLE")
log_results_df

In [ ]:
all_results_df = pd.concat([results_df, log_results_df], ignore_index=True).sort_values("RMSLE")
all_results_df

로그 변환을 적용한 모델과 적용하지 않은 모델을 함께 비교하였다. RMSLE는 로그 기반 평가 지표이므로 로그 변환 모델이 더 안정적인 성능을 보일 수 있다.

최종 모델은 모든 실험 결과 중 검증 RMSLE가 가장 낮은 모델로 선정한다.

## 19. 최종 모델 선택 및 전체 데이터 학습

검증 데이터에서 RMSLE가 가장 낮은 모델을 최종 모델로 선택한다. 모델 비교는 train 데이터를 학습용과 검증용으로 나누어 진행했지만, 최종 제출 파일을 만들 때는 전체 train 데이터를 사용하여 다시 학습한다.

전체 train 데이터를 사용하면 모델이 더 많은 데이터를 학습할 수 있으므로, 최종 제출 성능을 높이는 데 도움이 된다.

In [ ]:
best_row = all_results_df.iloc[0]
best_name = best_row["model"]

print("최종 선택 모델:", best_name)
print(best_row)

In [ ]:
use_log_target = best_name.endswith("_log_target")
base_name = best_name.replace("_log_target", "")

selected_base_model = models[base_name]

if use_log_target:
    selected_model = TransformedTargetRegressor(
        regressor=selected_base_model,
        func=np.log1p,
        inverse_func=np.expm1
    )
else:
    selected_model = selected_base_model

final_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", selected_model)
])

final_model.fit(X, y)

print("최종 모델 전체 train 데이터 학습 완료")
print("사용 모델:", best_name)

## 20. Test 데이터 예측

전체 train 데이터로 학습한 최종 모델을 이용하여 test 데이터의 `Rings` 값을 예측한다. RMSLE는 음수 예측값을 허용하지 않으므로, 예측값이 0보다 작을 경우 0으로 보정한다.

예측값의 분포를 확인하여 train 데이터의 `Rings` 분포와 크게 어긋나지 않는지도 함께 살펴본다.

In [ ]:
test_pred = final_model.predict(X_test)
test_pred = np.maximum(test_pred, 0)

test_pred[:10]

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(test_pred, bins=30)
plt.title("Distribution of Predicted Rings")
plt.xlabel("Predicted Rings")
plt.ylabel("Count")
plt.show()

In [ ]:
pd.Series(test_pred).describe()

예측값 분포를 통해 모델이 비정상적으로 큰 값이나 작은 값을 과도하게 예측하지 않는지 확인한다. 예측값이 대부분 현실적인 `Rings` 범위 안에 분포한다면 제출 파일을 생성할 수 있다.

## 21. 제출 파일 생성

Kaggle 제출 파일은 `sample_submission.csv`와 동일한 형식을 가져야 한다. 따라서 `id` 컬럼은 test 데이터의 id를 그대로 사용하고, `Rings` 컬럼에는 모델의 예측값을 넣는다.

생성된 제출 파일은 `submission.csv`로 저장한다.

In [ ]:
submission = sample_submission.copy()
submission["Rings"] = test_pred

submission.head()

In [ ]:
submission.to_csv("submission.csv", index=False)

print(submission.shape)

In [ ]:
pd.read_csv("submission.csv").head()

In [ ]:
pd.read_csv("submission.csv").info()

## 22. 답안 제출 및 결과 분석

생성한 `submission.csv` 파일을 Kaggle Late Submission에 제출하였다. 제출 파일은 `id`와 `Rings` 두 컬럼으로 구성되어 있으며, `sample_submission.csv`와 동일한 형식을 따른다.

최종 제출에는 검증 데이터에서 RMSLE가 가장 낮았던 모델을 사용하였다.

- 제출 파일명: submission.csv
- 사용 모델: 여기에 최종 선택 모델명 작성
- 검증 데이터 RMSLE: 여기에 검증 RMSLE 작성
- Kaggle 제출 RMSLE: 여기에 Kaggle 제출 점수 작성

검증 점수와 Kaggle 제출 점수는 완전히 같지 않을 수 있다. 검증 점수는 train 데이터 일부를 나누어 계산한 결과이고, Kaggle 점수는 별도의 test 데이터에 대해 계산된 결과이기 때문이다.

검증 점수와 제출 점수의 차이가 크지 않다면 모델이 학습 데이터에만 과하게 맞춰진 것이 아니라, 새로운 데이터에서도 비교적 안정적으로 예측했다고 볼 수 있다. 반대로 Kaggle 제출 점수가 검증 점수보다 크게 나쁘다면 과적합 가능성이나 검증 데이터 분리 방식의 한계를 고려해야 한다.

## 23. 결론

Regression with an Abalone Dataset 대회를 통해 정형 데이터 회귀 문제의 전체 과정을 수행하였다. 먼저 데이터 구조를 확인하고, 결측치 여부, 목표 변수 `Rings`의 분포, 범주형 변수 `Sex`, 수치형 변수들과 `Rings`의 관계를 분석하였다.

EDA 결과 전복의 길이, 지름, 높이, 무게 관련 변수들이 `Rings` 예측에 중요한 역할을 할 수 있음을 확인하였다. `Sex` 역시 전복의 성장 상태와 관련된 범주형 정보로 활용할 수 있었다.

모델링 단계에서는 Linear Regression, Ridge Regression, Random Forest Regressor, HistGradientBoostingRegressor를 비교하였다. 선형 모델은 baseline으로 사용하였고, 트리 기반 앙상블 모델은 비선형 관계와 변수 간 상호작용을 학습하기 위해 사용하였다.

평가 지표가 RMSLE이므로 목표 변수에 로그 변환을 적용한 모델도 함께 실험하였다. 로그 변환은 예측값과 실제값의 비율 차이를 줄이는 데 도움이 될 수 있으며, RMSLE 기반 회귀 문제에서 적합한 접근이다.

최종적으로 검증 데이터에서 RMSLE가 가장 낮은 모델을 선택하고, 전체 train 데이터로 다시 학습하였다. 이후 test 데이터의 `Rings` 값을 예측하여 `submission.csv` 파일을 생성하고 Kaggle에 제출하였다.

이번 과제를 통해 Kaggle 대회의 기본 참여 과정, EDA, 모델 선정, 평가 지표에 맞춘 학습 전략, 제출 파일 생성 과정을 정리할 수 있었다. 특히 대회의 평가 방식이 모델 선택과 전처리 방식에 직접적인 영향을 준다는 점을 확인하였다.